# Round 2 — Invest & Expand (Manual)

Allocate percentages `x` (research), `y` (scale), `z` (speed) with `x+y+z <= 100`.
Budget = 50,000 XIRECs, so `budget_used = 500 * (x+y+z)`.

- `research(x) = 200_000 * log(1+x) / log(101)`  — log, 0→200k
- `scale(y)   = 7 * y / 100`                     — linear, 0→7
- `speed_mult(z)` rank-based in [0.1, 0.9]. Model here: opponents ~Uniform(0, 100)
  ⇒ `speed_mult(z) = 0.1 + 0.8 * z/100` (tweak the opponent model as needed)

`PnL = research(x) * scale(y) * speed_mult(z) - budget_used`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

plt.rcParams.update({
    "figure.figsize": (10, 7),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "monospace",
})

print("Ready.")

## 1. Define objective

In [ ]:
BUDGET = 50_000
SPEED_MIN, SPEED_MAX = 0.1, 0.9

def research(x):
    return 200_000 * np.log(1 + x) / np.log(1 + 100)

def scale(y):
    return 7 * y / 100

def speed_mult(z):
    # Opponents ~ Uniform(0, 100): rank-percentile = z/100
    return SPEED_MIN + (SPEED_MAX - SPEED_MIN) * (z / 100)

def pnl(x, y, z):
    budget_used = BUDGET * (x + y + z) / 100
    return research(x) * scale(y) * speed_mult(z) - budget_used

## 2. Variable ranges

In [ ]:
GRID = np.linspace(0, 100, 101)  # resolution for x, y, z (in %)

## 3. Full 3D grid search

In [ ]:
xxx, yyy, zzz = np.meshgrid(GRID, GRID, GRID, indexing="ij")
feasible = (xxx + yyy + zzz) <= 100
F = np.where(feasible, pnl(xxx, yyy, zzz), np.nan)

idx = np.unravel_index(np.nanargmax(F), F.shape)
x_opt, y_opt, z_opt = xxx[idx], yyy[idx], zzz[idx]
pnl_opt = F[idx]
print(f"Optimum: x*={x_opt:.1f}%, y*={y_opt:.1f}%, z*={z_opt:.1f}%  |  speed_mult(z*)={speed_mult(z_opt):.3f}")
print(f"  budget_used = {BUDGET * (x_opt+y_opt+z_opt)/100:,.0f}")
print(f"  PnL        = {pnl_opt:,.0f}")

## 4. PnL surface sliced at optimal z*

In [ ]:
xx, yy = np.meshgrid(GRID, GRID)
feasible_slice = (xx + yy + z_opt) <= 100
ff = np.where(feasible_slice, pnl(xx, yy, z_opt), np.nan)

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(xx, yy, ff, cmap="viridis", edgecolor="none")
ax.scatter([x_opt], [y_opt], [pnl_opt], color="red", s=60, label=f"optimum ({pnl_opt:,.0f})")
ax.set_xlabel("research % (x)")
ax.set_ylabel("scale % (y)")
ax.set_zlabel("PnL")
ax.set_title(f"PnL surface at z*={z_opt:.1f}% (speed_mult={speed_mult(z_opt):.3f})")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Sensitivity: how the optimum shifts with opponent aggressiveness

Model the top-rank threshold as `z_top` — opponents ~Uniform(0, `z_top`). If `z_top` is small,
opponents barely invest in speed, so a little z goes a long way. If `z_top = 100`, it's the
default uniform-aggressive case.

In [ ]:
Z_TOP_GRID = np.linspace(10, 100, 46)  # opponent aggressiveness: opponents ~Uniform(0, z_top)

opt_x, opt_y, opt_z, opt_pnl = [], [], [], []
for z_top in Z_TOP_GRID:
    def sm(z, zt=z_top):
        return SPEED_MIN + (SPEED_MAX - SPEED_MIN) * np.minimum(z / zt, 1.0)
    F_zt = np.where(feasible, research(xxx) * scale(yyy) * sm(zzz) - BUDGET*(xxx+yyy+zzz)/100, np.nan)
    i = np.unravel_index(np.nanargmax(F_zt), F_zt.shape)
    opt_x.append(xxx[i]); opt_y.append(yyy[i]); opt_z.append(zzz[i]); opt_pnl.append(F_zt[i])

opt_x = np.array(opt_x); opt_y = np.array(opt_y); opt_z = np.array(opt_z); opt_pnl = np.array(opt_pnl)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.plot(Z_TOP_GRID, opt_x, label="research % (x*)")
ax1.plot(Z_TOP_GRID, opt_y, label="scale % (y*)")
ax1.plot(Z_TOP_GRID, opt_z, label="speed % (z*)")
ax1.set_xlabel("opponent aggressiveness (z_top)")
ax1.set_ylabel("optimal allocation %")
ax1.set_title("Optimal allocation vs opponent aggressiveness")
ax1.legend()

ax2.plot(Z_TOP_GRID, opt_pnl, color="C3")
ax2.set_xlabel("opponent aggressiveness (z_top)")
ax2.set_ylabel("max PnL")
ax2.set_title("Max PnL vs opponent aggressiveness")
ax2.axhline(0, color="gray", linestyle=":", linewidth=0.8)

plt.tight_layout()
plt.show()